# EDA: CRM vs Calendar events

In [1]:
import json, re
import pandas as pd
from pathlib import Path

pd.set_option("display.width", 200)
pd.set_option("display.max_columns", 30)
pd.set_option("display.max_colwidth", 60)

DATA = Path("../data")
crm = pd.DataFrame(json.loads((DATA / "crm_events.json").read_text()))
cal = pd.DataFrame(json.loads((DATA / "calendar_events.json").read_text()))
crm.shape, cal.shape

((20, 12), (22, 11))

In [2]:
crm.head(3)

,crm_id,subject,client_name,client_company,relationship_owner,meeting_date,meeting_time,meeting_type,location,notes,status,created_at
0,CRM-1001,Q1 Portfolio Review,David Park,Meridian Capital,Sarah Chen,2025-03-10,14:00,In-Person,HQ - Conference Room B,Review Q1 allocation strategy. David requested updated f...,Completed,2025-02-28T09:15:00Z
1,CRM-1002,Portfolio Walkthrough,Mark Johnson,Summit Advisors,Sarah Chen,2025-03-12,10:00,In-Person,NYC Office - 30th Floor,Walk through current portfolio positioning and Q2 outlook.,Confirmed,2025-03-01T11:30:00Z
2,CRM-1003,Introductory Call,Rachel Torres,Lakeshore Partners,James Wu,2025-03-14,15:00,Virtual,NaN,First touch. Referred by Tom Nakamura at Bridgepoint.,Scheduled,2025-03-10T14:00:00Z


In [3]:
cal.head(3)

,event_id,title,organizer,attendees,start_time,end_time,location,description,is_recurring,status,created_at
0,CAL-A1,Q1 Portfolio Review - Meridian Capital,sarah.chen@firma.com,"[sarah.chen@firma.com, david.park@meridiancap.com]",2025-03-10T14:00:00,2025-03-10T15:30:00,Conference Room B,Quarterly review of portfolio allocation and performance.,False,confirmed,2025-02-27T10:00:00Z
1,CAL-A2,Summit Advisors - Portfolio Discussion,sarah.chen@firma.com,"[sarah.chen@firma.com, mark.johnson@summitadv.com, anna....",2025-03-12T10:00:00,2025-03-12T11:00:00,Zoom - https://zoom.us/j/98765432100,Portfolio walkthrough with Summit team.,False,confirmed,2025-02-28T15:45:00Z
2,CAL-A3,Weekly Team Sync,sarah.chen@firma.com,"[sarah.chen@firma.com, james.wu@firma.com, priya.sharma@...",2025-03-11T09:00:00,2025-03-11T09:30:00,Virtual - Microsoft Teams,Standing weekly sync.,True,confirmed,2025-01-06T08:00:00Z


## Schema and nulls

In [4]:
pd.concat({"crm": crm.dtypes, "cal": cal.dtypes}, axis=1)

,crm,cal
crm_id,str,NaN
subject,str,NaN
client_name,str,NaN
client_company,str,NaN
relationship_owner,str,NaN
meeting_date,str,NaN
meeting_time,str,NaN
meeting_type,str,NaN
location,str,str
notes,str,NaN


In [5]:
pd.concat({"crm_nulls": crm.isna().sum(), "cal_nulls": cal.isna().sum()}, axis=1).astype("Int64")

,crm_nulls,cal_nulls
crm_id,0,<NA>
subject,0,<NA>
client_name,4,<NA>
client_company,4,<NA>
relationship_owner,0,<NA>
meeting_date,0,<NA>
meeting_time,1,<NA>
meeting_type,0,<NA>
location,4,2
notes,0,<NA>


In [6]:
# nulls concentrated in a few rows
crm[crm.isna().any(axis=1)][["crm_id", "subject", "client_name", "meeting_time", "location", "meeting_type"]]

,crm_id,subject,client_name,meeting_time,location,meeting_type
2,CRM-1003,Introductory Call,Rachel Torres,15:00,NaN,Virtual
5,CRM-1006,Pipeline Review,NaN,16:00,HQ - Room 4A,Internal
6,CRM-1007,Follow-up Discussion,David Park,NaN,NaN,Virtual
8,CRM-1009,Board Prep Session,NaN,09:00,HQ - Boardroom,Internal
12,CRM-1013,LP Advisory Committee Prep,NaN,10:00,HQ - Boardroom,Internal
13,CRM-1014,Quick Check-in,Michael Reeves,08:30,NaN,Virtual
17,CRM-1018,Follow-up Call,Patricia Langley,16:30,NaN,Virtual
18,CRM-1019,Quarterly Business Review,NaN,14:00,HQ - Large Conference Room,Internal


In [7]:
cal[cal.isna().any(axis=1)][["event_id", "title", "attendees", "location", "description", "status"]]

,event_id,title,attendees,location,description,status
10,CAL-A11,Client Reception,[],NaN,NaN,tentative
14,CAL-A15,Michael Reeves - Quick Sync,"[james.wu@firma.com, michael.reeves@evergreencap.com]",NaN,Capital call timing discussion.,confirmed


In [8]:
# empty strings hiding as non-null
(crm == "").sum()[lambda s: s > 0], (cal == "").sum()[lambda s: s > 0]

(notes    1
 dtype: int64,
 Series([], dtype: int64))

## Categoricals

In [9]:
# different vocab and casing per source; crm has lifecycle states, cal only confirmed/tentative
crm["status"].value_counts(), cal["status"].value_counts()

(status
 Confirmed    14
 Scheduled     2
 Tentative     2
 Completed     1
 Cancelled     1
 Name: count, dtype: int64,
 status
 confirmed    21
 tentative     1
 Name: count, dtype: int64)

In [10]:
crm["meeting_type"].value_counts(dropna=False)

meeting_type
In-Person    9
Virtual      7
Internal     4
Name: count, dtype: int64

In [11]:
# priya organizes but never owns; owner != organizer is possible
crm["relationship_owner"].value_counts(), cal["organizer"].value_counts()

(relationship_owner
 Sarah Chen    11
 James Wu       9
 Name: count, dtype: int64,
 organizer
 sarah.chen@firma.com      13
 james.wu@firma.com         7
 priya.sharma@firma.com     2
 Name: count, dtype: int64)

## CRM dates

In [12]:
parsed = pd.to_datetime(crm["meeting_date"], format="%Y-%m-%d", errors="coerce")
crm.loc[parsed.isna(), ["crm_id", "subject", "meeting_date", "created_at"]]

,crm_id,subject,meeting_date,created_at
7,CRM-1008,Lunch Meeting,03-15/2025,2025-03-08T13:00:00Z


In [13]:
# one row uses MM-DD/YYYY; accept it explicitly rather than guessing
alt = pd.to_datetime(crm["meeting_date"], format="%m-%d/%Y", errors="coerce")
crm["date"] = parsed.fillna(alt)
crm["date"].isna().sum(), crm["date"].min(), crm["date"].max()

(np.int64(0),
 Timestamp('2025-03-10 00:00:00'),
 Timestamp('2025-04-02 00:00:00'))

In [14]:
crm["start"] = pd.to_datetime(crm["date"].dt.strftime("%Y-%m-%d") + " " + crm["meeting_time"], errors="coerce")
crm.loc[crm["start"].isna(), ["crm_id", "subject", "date", "meeting_time"]]

,crm_id,subject,date,meeting_time
6,CRM-1007,Follow-up Discussion,2025-03-18,NaN


In [15]:
crm["dow"] = crm["date"].dt.day_name()
crm[crm["dow"].isin(["Saturday", "Sunday"])][["crm_id", "subject", "client_company", "date", "dow"]]

,crm_id,subject,client_company,date,dow
7,CRM-1008,Lunch Meeting,Atlas Ventures,2025-03-15,Saturday


## Calendar timestamps

In [16]:
ts = cal[["event_id", "start_time", "end_time"]].copy()
ts["start_len"] = ts["start_time"].str.len()
ts["end_len"] = ts["end_time"].str.len()
ts["start_tz"] = ts["start_time"].str.endswith("Z")
ts["end_tz"] = ts["end_time"].str.endswith("Z")
ts.groupby(["start_len", "end_len", "start_tz", "end_tz"]).size()

start_len  end_len  start_tz  end_tz
19         16       False     False      1
           19       False     False     20
20         20       True      True       1
dtype: int64

In [17]:
ts[(ts["start_tz"]) | (ts["end_len"] != 19)]

,event_id,start_time,end_time,start_len,end_len,start_tz,end_tz
3,CAL-A4,2025-03-13T19:00:00Z,2025-03-13T20:30:00Z,20,20,True,True
10,CAL-A11,2025-03-14T18:00:00,2025-03-14T20:00,19,16,False,False


In [18]:
# mixed naive/aware: naive values are local (ET); the Z row needs converting, not stripping
def to_local(s):
    t = pd.to_datetime(s, errors="coerce", utc=False)
    if t.tzinfo is not None:
        return t.tz_convert("America/New_York").tz_localize(None)
    return t

cal["start"] = cal["start_time"].map(to_local)
cal["end"] = cal["end_time"].map(to_local)
cal.loc[cal["event_id"].isin(["CAL-A4", "CAL-A11"]), ["event_id", "start_time", "start", "end_time", "end"]]

,event_id,start_time,start,end_time,end
3,CAL-A4,2025-03-13T19:00:00Z,2025-03-13 15:00:00,2025-03-13T20:30:00Z,2025-03-13 16:30:00
10,CAL-A11,2025-03-14T18:00:00,2025-03-14 18:00:00,2025-03-14T20:00,2025-03-14 20:00:00


In [19]:
# 19:00Z on 2025-03-13 is 15:00 EDT; a fixed UTC-5 assumption would give 14:00
pd.Timestamp("2025-03-13T19:00:00Z").tz_convert("America/New_York"), pd.Timestamp("2025-03-13T19:00:00Z").tz_convert("Etc/GMT+5")

(Timestamp('2025-03-13 15:00:00-0400', tz='America/New_York'),
 Timestamp('2025-03-13 14:00:00-0500', tz='Etc/GMT+5'))

In [20]:
cal["duration_min"] = (cal["end"] - cal["start"]).dt.total_seconds() / 60
cal["dow"] = cal["start"].dt.day_name()
cal[cal["dow"].isin(["Saturday", "Sunday"])][["event_id", "title", "start", "dow"]]

,event_id,title,start,dow
8,CAL-A9,Atlas Ventures Lunch,2025-03-15 12:00:00,Saturday


In [21]:
# recurring entries are series instances, not a single meeting
cal[cal["is_recurring"]][["event_id", "title", "start", "organizer"]]

,event_id,title,start,organizer
2,CAL-A3,Weekly Team Sync,2025-03-11 09:00:00,sarah.chen@firma.com
6,CAL-A7,Internal Pipeline Review,2025-03-14 16:00:00,sarah.chen@firma.com
17,CAL-A18,Weekly Team Sync,2025-03-18 09:00:00,sarah.chen@firma.com


In [22]:
# same organizer, overlapping time windows
c = cal[["event_id", "title", "organizer", "start", "end"]]
ov = c.merge(c, on="organizer", suffixes=("_a", "_b"))
ov = ov[(ov["event_id_a"] < ov["event_id_b"]) & (ov["start_a"] < ov["end_b"]) & (ov["start_b"] < ov["end_a"])]
ov[["organizer", "event_id_a", "title_a", "start_a", "end_a", "event_id_b", "title_b", "start_b", "end_b"]]

,organizer,event_id_a,title_a,start_a,end_a,event_id_b,title_b,start_b,end_b
53,james.wu@firma.com,CAL-A5,Investor Update - Pinnacle,2025-03-17 11:00:00,2025-03-17 12:00:00,CAL-A6,Pinnacle Group - Q1 Update,2025-03-17 11:30:00,2025-03-17 12:30:00


## Attendees

In [23]:
att = cal[["event_id", "attendees"]].explode("attendees")
att["n"] = att.groupby("event_id")["attendees"].transform("count")
att[att["n"] == 0]

,event_id,attendees,n
10,CAL-A11,NaN,0


In [24]:
EMAIL = r"^[\w.+-]+@[\w-]+(\.[\w-]+)+$"
bad = att[att["attendees"].notna() & ~att["attendees"].str.match(EMAIL)]
bad

,event_id,attendees,n
15,CAL-A16,raj.patel[at]atlasvc.com,3
19,CAL-A20,external-guests,6


In [25]:
att["domain"] = att["attendees"].str.extract(r"@(.+)$")[0]
att["domain"].value_counts()

domain
firma.com              43
pinnaclegp.com          4
meridiancap.com         2
summitadv.com           2
crestviewhold.com       2
atlasvc.com             2
horizonwp.com           2
granitepointcap.com     2
evergreencap.com        1
redwoodinst.com         1
Name: count, dtype: int64

In [26]:
# external domain(s) per event ~ client company
ext = att[att["domain"].notna() & (att["domain"] != "firma.com")].groupby("event_id")["domain"].agg(lambda s: sorted(set(s)))
cal = cal.merge(ext.rename("ext_domains"), left_on="event_id", right_index=True, how="left")
cal[["event_id", "title", "ext_domains"]]

,event_id,title,ext_domains
0,CAL-A1,Q1 Portfolio Review - Meridian Capital,[meridiancap.com]
1,CAL-A2,Summit Advisors - Portfolio Discussion,[summitadv.com]
2,CAL-A3,Weekly Team Sync,NaN
3,CAL-A4,Crestview Holdings - DD Meeting,[crestviewhold.com]
4,CAL-A5,Investor Update - Pinnacle,[pinnaclegp.com]
5,CAL-A6,Pinnacle Group - Q1 Update,[pinnaclegp.com]
6,CAL-A7,Internal Pipeline Review,NaN
7,CAL-A8,Meridian Capital - Follow-up,[meridiancap.com]
8,CAL-A9,Atlas Ventures Lunch,[atlasvc.com]
9,CAL-A10,Board Prep - Q1 Materials Review,NaN


In [27]:
# one event has several client firms; a single client_company can't represent it
cal[cal["ext_domains"].map(lambda d: isinstance(d, list) and len(d) > 1)][["event_id", "title", "ext_domains"]]

,event_id,title,ext_domains
19,CAL-A20,Fund VII Closing Dinner,"[granitepointcap.com, horizonwp.com, pinnaclegp.com]"


In [28]:
# internal-only events: cal has more of them than crm
cal["internal"] = cal["attendees"].map(lambda a: len(a) > 0 and all(e.endswith("firma.com") for e in a))
cal[cal["internal"]][["event_id", "title", "start"]], crm[crm["meeting_type"] == "Internal"][["crm_id", "subject", "start"]]

(   event_id                             title               start
 2    CAL-A3                  Weekly Team Sync 2025-03-11 09:00:00
 6    CAL-A7          Internal Pipeline Review 2025-03-14 16:00:00
 9   CAL-A10  Board Prep - Q1 Materials Review 2025-03-19 09:00:00
 13  CAL-A14         LPAC Prep Working Session 2025-03-24 10:00:00
 17  CAL-A18                  Weekly Team Sync 2025-03-18 09:00:00
 18  CAL-A19            Fund VII Roadshow Prep 2025-03-28 14:00:00
 21  CAL-A22                     QBR - Q1 Wrap 2025-04-01 14:00:00,
       crm_id                     subject               start
 5   CRM-1006             Pipeline Review 2025-03-14 16:00:00
 8   CRM-1009          Board Prep Session 2025-03-19 09:00:00
 12  CRM-1013  LP Advisory Committee Prep 2025-03-24 10:00:00
 18  CRM-1019   Quarterly Business Review 2025-04-01 14:00:00)

## Names vs emails

In [29]:
# crm names -> first.last; apostrophes and "Multiple" break the mapping
local_parts = set(att["attendees"].dropna().str.extract(r"^([^@]+)@")[0].dropna())
def name_key(n):
    if not isinstance(n, str):
        return None
    parts = re.sub(r"[^a-z ]", "", n.lower()).split()
    return ".".join(parts) if len(parts) == 2 else None

crm["client_key"] = crm["client_name"].map(name_key)
crm["client_in_cal"] = crm["client_key"].isin(local_parts)
crm[["crm_id", "client_name", "client_key", "client_in_cal"]]

,crm_id,client_name,client_key,client_in_cal
0,CRM-1001,David Park,david.park,True
1,CRM-1002,Mark Johnson,mark.johnson,True
2,CRM-1003,Rachel Torres,rachel.torres,False
3,CRM-1004,Patricia Langley,patricia.langley,True
4,CRM-1005,Kevin O'Brien,kevin.obrien,True
5,CRM-1006,NaN,NaN,False
6,CRM-1007,David Park,david.park,True
7,CRM-1008,Lisa Tran,lisa.tran,True
8,CRM-1009,NaN,NaN,False
9,CRM-1010,Alan Whitfield,alan.whitfield,False


In [30]:
crm[crm["client_name"].str.contains(r"[^A-Za-z ]", na=False) | (crm["client_name"] == "Multiple")][["crm_id", "client_name", "client_key"]]

,crm_id,client_name,client_key
4,CRM-1005,Kevin O'Brien,kevin.obrien
16,CRM-1017,Multiple,NaN


In [31]:
# clients with no calendar footprint at all
crm[~crm["client_in_cal"] & crm["client_name"].notna()][["crm_id", "subject", "client_name", "client_company", "start"]]

,crm_id,subject,client_name,client_company,start
2,CRM-1003,Introductory Call,Rachel Torres,Lakeshore Partners,2025-03-14 15:00:00
9,CRM-1010,Strategy Discussion,Alan Whitfield,Northwind Investments,2025-03-20 10:00:00
16,CRM-1017,Closing Dinner,Multiple,Fund VII Investors,2025-03-28 19:00:00
19,CRM-1020,Northwind - Fund VII Interest,Alan Whitfield,Northwind Investments,2025-04-02 11:00:00


## Locations

In [32]:
# same place, different spellings across sources
loc = pd.concat([
    crm[["location"]].assign(src="crm"),
    cal[["location"]].assign(src="cal"),
]).dropna()
loc.groupby("location")["src"].agg(lambda s: sorted(s)).to_frame()

,src
location,
Boston Office,"[cal, crm]"
Boston Office - Room 301,[cal]
Conference Room A,[cal]
Conference Room B,[cal]
DC Office - Main Conference Room,"[cal, crm]"
HQ - Boardroom,"[cal, cal, crm, crm]"
HQ - Conference Room A,[crm]
HQ - Conference Room B,[crm]
HQ - Large Conference Room,"[cal, crm]"


In [33]:
VIRTUAL = ("zoom", "teams", "virtual", "meet", "webex")
def loc_kind(loc, mtype=None):
    if mtype == "Virtual" or (isinstance(loc, str) and any(k in loc.lower() for k in VIRTUAL)):
        return "virtual"
    return "in_person" if isinstance(loc, str) else None

crm["loc_kind"] = [loc_kind(l, t) for l, t in zip(crm["location"], crm["meeting_type"])]
cal["loc_kind"] = [loc_kind(l) for l in cal["location"]]
crm["loc_kind"].value_counts(dropna=False), cal["loc_kind"].value_counts(dropna=False)

(loc_kind
 in_person    13
 virtual       7
 Name: count, dtype: int64,
 loc_kind
 in_person    13
 virtual       7
 NaN           2
 Name: count, dtype: int64)

## Cross-source matching

In [34]:
crm["day"] = crm["date"].dt.date
cal["day"] = cal["start"].dt.date
pd.concat([crm["day"].value_counts().rename("crm"), cal["day"].value_counts().rename("cal")], axis=1).fillna(0).astype(int).sort_index()

,crm,cal
day,,
2025-03-10,1,1
2025-03-11,0,1
2025-03-12,1,1
2025-03-13,1,1
2025-03-14,2,2
2025-03-15,1,1
2025-03-17,1,2
2025-03-18,1,2
2025-03-19,1,1


In [35]:
# no shared id; try the obvious key first: same calendar day + same person
crm["owner_email"] = crm["relationship_owner"].str.lower().str.replace(" ", ".") + "@firma.com"

cand = crm.merge(cal, left_on=["day", "owner_email"], right_on=["day", "organizer"], suffixes=("_crm", "_cal"))
cand["dt_min"] = (cand["start_cal"] - cand["start_crm"]).dt.total_seconds() / 60
cand[["crm_id", "subject", "event_id", "title", "start_crm", "start_cal", "dt_min"]].sort_values("start_crm")

,crm_id,subject,event_id,title,start_crm,start_cal,dt_min
0,CRM-1001,Q1 Portfolio Review,CAL-A1,Q1 Portfolio Review - Meridian Capital,2025-03-10 14:00:00,2025-03-10 14:00:00,0.0
1,CRM-1002,Portfolio Walkthrough,CAL-A2,Summit Advisors - Portfolio Discussion,2025-03-12 10:00:00,2025-03-12 10:00:00,0.0
2,CRM-1004,Due Diligence Review,CAL-A4,Crestview Holdings - DD Meeting,2025-03-13 14:00:00,2025-03-13 15:00:00,60.0
5,CRM-1006,Pipeline Review,CAL-A7,Internal Pipeline Review,2025-03-14 16:00:00,2025-03-14 16:00:00,0.0
8,CRM-1008,Lunch Meeting,CAL-A9,Atlas Ventures Lunch,2025-03-15 12:00:00,2025-03-15 12:00:00,0.0
3,CRM-1005,Investor Update,CAL-A5,Investor Update - Pinnacle,2025-03-17 11:00:00,2025-03-17 11:00:00,0.0
4,CRM-1005,Investor Update,CAL-A6,Pinnacle Group - Q1 Update,2025-03-17 11:00:00,2025-03-17 11:30:00,30.0
9,CRM-1009,Board Prep Session,CAL-A10,Board Prep - Q1 Materials Review,2025-03-19 09:00:00,2025-03-19 09:00:00,0.0
10,CRM-1011,Annual Allocation Review,CAL-A12,Horizon Wealth - Year-End Review,2025-03-21 09:00:00,2025-03-21 09:00:00,0.0
11,CRM-1012,Co-Investment Opportunity,CAL-A13,Granite Point - Co-Invest Discussion,2025-03-21 14:00:00,2025-03-21 14:00:00,0.0


In [36]:
# some pairs are pure day+owner coincidences; need another signal
def norm(s):
    return re.sub(r"[^a-z0-9 ]", " ", str(s).lower()).split()

STOP = {"the", "and", "a", "of", "with", "for", "re", "q1", "review", "meeting", "call", "session", "discussion"}
def overlap(a, b):
    ta, tb = set(norm(a)) - STOP, set(norm(b)) - STOP
    return len(ta & tb) / max(1, min(len(ta), len(tb)))

cand["company_tok"] = cand["client_company"].fillna("") + " " + cand["subject"] + " " + cand["notes"].fillna("")
cand["cal_tok"] = cand["title"] + " " + cand["description"].fillna("")
cand["text_sim"] = [overlap(a, b) for a, b in zip(cand["company_tok"], cand["cal_tok"])]

def company_domain(c):
    return re.sub(r"[^a-z]", "", str(c).lower().split()[0]) if isinstance(c, str) else ""
cand["domain_hit"] = [
    any(company_domain(c) and company_domain(c)[:6] in d for d in (ds if isinstance(ds, list) else []))
    for c, ds in zip(cand["client_company"], cand["ext_domains"])
]
cand[["crm_id", "event_id", "dt_min", "text_sim", "domain_hit", "client_company", "ext_domains"]].sort_values("crm_id")

,crm_id,event_id,dt_min,text_sim,domain_hit,client_company,ext_domains
0,CRM-1001,CAL-A1,0.0,0.666667,True,Meridian Capital,[meridiancap.com]
1,CRM-1002,CAL-A2,0.0,0.800000,True,Summit Advisors,[summitadv.com]
2,CRM-1004,CAL-A4,60.0,0.750000,True,Crestview Holdings,[crestviewhold.com]
3,CRM-1005,CAL-A5,0.0,1.000000,True,Pinnacle Group,[pinnaclegp.com]
4,CRM-1005,CAL-A6,30.0,0.571429,True,Pinnacle Group,[pinnaclegp.com]
5,CRM-1006,CAL-A7,0.0,0.250000,False,NaN,NaN
6,CRM-1007,CAL-A8,NaN,0.666667,True,Meridian Capital,[meridiancap.com]
7,CRM-1007,CAL-A18,NaN,0.000000,False,Meridian Capital,NaN
8,CRM-1008,CAL-A9,0.0,0.666667,True,Atlas Ventures,[atlasvc.com]
9,CRM-1009,CAL-A10,0.0,0.600000,False,NaN,NaN


In [37]:
cand["score"] = (
    cand["dt_min"].abs().fillna(60).clip(upper=180).rsub(180) / 180 * 0.4
    + cand["text_sim"] * 0.3
    + cand["domain_hit"] * 0.3
)
cand = cand.sort_values("score", ascending=False)
cand[["crm_id", "subject", "event_id", "title", "dt_min", "text_sim", "domain_hit", "score"]]

,crm_id,subject,event_id,title,dt_min,text_sim,domain_hit,score
3,CRM-1005,Investor Update,CAL-A5,Investor Update - Pinnacle,0.0,1.000000,True,1.000000
11,CRM-1012,Co-Investment Opportunity,CAL-A13,Granite Point - Co-Invest Discussion,0.0,0.900000,True,0.970000
1,CRM-1002,Portfolio Walkthrough,CAL-A2,Summit Advisors - Portfolio Discussion,0.0,0.800000,True,0.940000
16,CRM-1018,Follow-up Call,CAL-A21,Crestview Follow-up Call,0.0,0.777778,True,0.933333
10,CRM-1011,Annual Allocation Review,CAL-A12,Horizon Wealth - Year-End Review,0.0,0.777778,True,0.933333
0,CRM-1001,Q1 Portfolio Review,CAL-A1,Q1 Portfolio Review - Meridian Capital,0.0,0.666667,True,0.900000
8,CRM-1008,Lunch Meeting,CAL-A9,Atlas Ventures Lunch,0.0,0.666667,True,0.900000
13,CRM-1015,Infrastructure Fund Pitch,CAL-A16,Atlas Ventures - Infrastructure Deep Dive,0.0,0.666667,True,0.900000
12,CRM-1014,Quick Check-in,CAL-A15,Michael Reeves - Quick Sync,0.0,0.500000,True,0.850000
4,CRM-1005,Investor Update,CAL-A6,Pinnacle Group - Q1 Update,30.0,0.571429,True,0.804762


In [38]:
# greedy 1:1 assignment above a threshold
THRESH = 0.45
used_crm, used_cal, pairs = set(), set(), []
for _, r in cand.iterrows():
    if r["score"] < THRESH or r["crm_id"] in used_crm or r["event_id"] in used_cal:
        continue
    used_crm.add(r["crm_id"]); used_cal.add(r["event_id"]); pairs.append(r)
matched = pd.DataFrame(pairs)
matched[["crm_id", "subject", "event_id", "title", "dt_min", "score"]]

,crm_id,subject,event_id,title,dt_min,score
3,CRM-1005,Investor Update,CAL-A5,Investor Update - Pinnacle,0.0,1.000000
11,CRM-1012,Co-Investment Opportunity,CAL-A13,Granite Point - Co-Invest Discussion,0.0,0.970000
1,CRM-1002,Portfolio Walkthrough,CAL-A2,Summit Advisors - Portfolio Discussion,0.0,0.940000
16,CRM-1018,Follow-up Call,CAL-A21,Crestview Follow-up Call,0.0,0.933333
10,CRM-1011,Annual Allocation Review,CAL-A12,Horizon Wealth - Year-End Review,0.0,0.933333
0,CRM-1001,Q1 Portfolio Review,CAL-A1,Q1 Portfolio Review - Meridian Capital,0.0,0.900000
8,CRM-1008,Lunch Meeting,CAL-A9,Atlas Ventures Lunch,0.0,0.900000
13,CRM-1015,Infrastructure Fund Pitch,CAL-A16,Atlas Ventures - Infrastructure Deep Dive,0.0,0.900000
12,CRM-1014,Quick Check-in,CAL-A15,Michael Reeves - Quick Sync,0.0,0.850000
2,CRM-1004,Due Diligence Review,CAL-A4,Crestview Holdings - DD Meeting,60.0,0.791667


In [39]:
# rejected candidates: same day, same owner, different meeting
cand[~cand.index.isin(matched.index)][["crm_id", "subject", "event_id", "title", "dt_min", "score"]]

,crm_id,subject,event_id,title,dt_min,score
4,CRM-1005,Investor Update,CAL-A6,Pinnacle Group - Q1 Update,30.0,0.804762
7,CRM-1007,Follow-up Discussion,CAL-A18,Weekly Team Sync,NaN,0.266667


In [40]:
crm[~crm["crm_id"].isin(used_crm)][["crm_id", "subject", "relationship_owner", "client_company", "start", "status"]]

,crm_id,subject,relationship_owner,client_company,start,status
2,CRM-1003,Introductory Call,James Wu,Lakeshore Partners,2025-03-14 15:00:00,Scheduled
9,CRM-1010,Strategy Discussion,James Wu,Northwind Investments,2025-03-20 10:00:00,Tentative
12,CRM-1013,LP Advisory Committee Prep,Sarah Chen,NaN,2025-03-24 10:00:00,Confirmed
19,CRM-1020,Northwind - Fund VII Interest,James Wu,Northwind Investments,2025-04-02 11:00:00,Tentative


In [41]:
cal[~cal["event_id"].isin(used_cal)][["event_id", "title", "organizer", "start", "status", "is_recurring"]]

,event_id,title,organizer,start,status,is_recurring
2,CAL-A3,Weekly Team Sync,sarah.chen@firma.com,2025-03-11 09:00:00,confirmed,True
5,CAL-A6,Pinnacle Group - Q1 Update,james.wu@firma.com,2025-03-17 11:30:00,confirmed,False
10,CAL-A11,Client Reception,priya.sharma@firma.com,2025-03-14 18:00:00,tentative,False
13,CAL-A14,LPAC Prep Working Session,priya.sharma@firma.com,2025-03-24 10:00:00,confirmed,False
17,CAL-A18,Weekly Team Sync,sarah.chen@firma.com,2025-03-18 09:00:00,confirmed,True
18,CAL-A19,Fund VII Roadshow Prep,sarah.chen@firma.com,2025-03-28 14:00:00,confirmed,False


In [42]:
# LPAC prep sits unmatched on both sides at the same time; owner != organizer broke the join
left = crm[~crm["crm_id"].isin(used_crm)]
right = cal[~cal["event_id"].isin(used_cal)]
pass2 = left.merge(right, on="day", suffixes=("_crm", "_cal"))
pass2["dt_min"] = (pass2["start_cal"] - pass2["start_crm"]).dt.total_seconds() / 60
pass2["text_sim"] = [overlap(a + " " + str(n), b + " " + str(d)) for a, n, b, d in zip(pass2["subject"], pass2["notes"], pass2["title"], pass2["description"])]
pass2[["crm_id", "subject", "relationship_owner", "event_id", "title", "organizer", "dt_min", "text_sim"]]

,crm_id,subject,relationship_owner,event_id,title,organizer,dt_min,text_sim
0,CRM-1003,Introductory Call,James Wu,CAL-A11,Client Reception,priya.sharma@firma.com,180.0,0.000000
1,CRM-1013,LP Advisory Committee Prep,Sarah Chen,CAL-A14,LPAC Prep Working Session,priya.sharma@firma.com,0.0,0.555556


In [43]:
extra = pass2[(pass2["dt_min"].abs() <= 30) & (pass2["text_sim"] >= 0.3)]
for _, r in extra.iterrows():
    used_crm.add(r["crm_id"]); used_cal.add(r["event_id"])
matched = pd.concat([matched, extra.assign(score=0.5, domain_hit=False)], ignore_index=True)
extra[["crm_id", "event_id", "relationship_owner", "organizer", "dt_min"]]

,crm_id,event_id,relationship_owner,organizer,dt_min
1,CRM-1013,CAL-A14,Sarah Chen,priya.sharma@firma.com,0.0


In [44]:
crm[~crm["crm_id"].isin(used_crm)][["crm_id", "subject", "client_company", "start", "status"]]

,crm_id,subject,client_company,start,status
2,CRM-1003,Introductory Call,Lakeshore Partners,2025-03-14 15:00:00,Scheduled
9,CRM-1010,Strategy Discussion,Northwind Investments,2025-03-20 10:00:00,Tentative
19,CRM-1020,Northwind - Fund VII Interest,Northwind Investments,2025-04-02 11:00:00,Tentative


In [45]:
cal[~cal["event_id"].isin(used_cal)][["event_id", "title", "start", "status", "is_recurring"]]

,event_id,title,start,status,is_recurring
2,CAL-A3,Weekly Team Sync,2025-03-11 09:00:00,confirmed,True
5,CAL-A6,Pinnacle Group - Q1 Update,2025-03-17 11:30:00,confirmed,False
10,CAL-A11,Client Reception,2025-03-14 18:00:00,tentative,False
17,CAL-A18,Weekly Team Sync,2025-03-18 09:00:00,confirmed,True
18,CAL-A19,Fund VII Roadshow Prep,2025-03-28 14:00:00,confirmed,False


## Field conflicts within matched pairs

In [46]:
matched[matched["dt_min"] != 0][["crm_id", "event_id", "start_crm", "start_cal", "dt_min"]]

,crm_id,event_id,start_crm,start_cal,dt_min
9,CRM-1004,CAL-A4,2025-03-13 14:00:00,2025-03-13 15:00:00,60.0
10,CRM-1007,CAL-A8,NaT,2025-03-18 15:00:00,NaN
11,CRM-1016,CAL-A17,2025-03-27 13:00:00,2025-03-27 15:00:00,120.0


In [47]:
k = matched[["loc_kind_crm", "loc_kind_cal"]].notna().all(axis=1)
matched[k & (matched["loc_kind_crm"] != matched["loc_kind_cal"])][["crm_id", "event_id", "meeting_type", "location_crm", "location_cal"]]

,crm_id,event_id,meeting_type,location_crm,location_cal
2,CRM-1002,CAL-A2,In-Person,NYC Office - 30th Floor,Zoom - https://zoom.us/j/98765432100


In [48]:
matched[["crm_id", "event_id", "location_crm", "location_cal"]][matched["location_crm"].fillna("") != matched["location_cal"].fillna("")]

,crm_id,event_id,location_crm,location_cal
0,CRM-1005,CAL-A5,Boston Office,Boston Office - Room 301
1,CRM-1012,CAL-A13,Zoom,Zoom - https://zoom.us/j/11223344
2,CRM-1002,CAL-A2,NYC Office - 30th Floor,Zoom - https://zoom.us/j/98765432100
3,CRM-1018,CAL-A21,NaN,Zoom
4,CRM-1011,CAL-A12,NYC Office,NYC Office - 12th Floor
5,CRM-1001,CAL-A1,HQ - Conference Room B,Conference Room B
7,CRM-1015,CAL-A16,HQ - Conference Room A,Conference Room A
10,CRM-1007,CAL-A8,NaN,Zoom
12,CRM-1017,CAL-A20,The Palm - DC,The Palm Restaurant


In [49]:
matched["status_crm_n"] = matched["status_crm"].str.lower()
matched["status_cal_n"] = matched["status_cal"].str.lower()
matched[matched["status_crm_n"] != matched["status_cal_n"]][["crm_id", "event_id", "status_crm", "status_cal", "notes"]]

,crm_id,event_id,status_crm,status_cal,notes
5,CRM-1001,CAL-A1,Completed,confirmed,Review Q1 allocation strategy. David requested updated f...
10,CRM-1007,CAL-A8,Scheduled,confirmed,Follow up on fee schedule questions from Q1 review.
14,CRM-1009,CAL-A10,Cancelled,confirmed,Prep Q1 board materials. CANCELLED — rescheduled to 3/26.


In [50]:
# cancelled note points to 3/26; nothing board-related exists there in either source
crm[crm["notes"].str.contains(r"\d{1,2}/\d{1,2}", na=False)][["crm_id", "notes"]], cal[cal["day"].astype(str) == "2025-03-26"][["event_id", "title"]]

(      crm_id                                                        notes
 8   CRM-1009    Prep Q1 board materials. CANCELLED — rescheduled to 3/26.
 17  CRM-1018  Follow up on DD questions from 3/13 meeting. Patricia wa...,
    event_id                                      title
 15  CAL-A16  Atlas Ventures - Infrastructure Deep Dive)

In [51]:
# calendar entry usually created ~1 day before the crm one; ESG pair is reversed by ~2 days and also carries the 2h time conflict
fs = matched[["crm_id", "event_id", "created_at_crm", "created_at_cal"]].copy()
fs["cal_first"] = fs["created_at_cal"] < fs["created_at_crm"]
fs["gap_h"] = (pd.to_datetime(fs["created_at_crm"]) - pd.to_datetime(fs["created_at_cal"])).dt.total_seconds() / 3600
fs.sort_values("gap_h")

,crm_id,event_id,created_at_crm,created_at_cal,cal_first,gap_h
11,CRM-1016,CAL-A17,2025-03-18T09:45:00Z,2025-03-20T08:30:00Z,False,-46.750000
10,CRM-1007,CAL-A8,2025-03-11T09:30:00Z,2025-03-11T09:45:00Z,False,-0.250000
15,CRM-1006,CAL-A7,2025-01-15T10:00:00Z,2025-01-15T10:05:00Z,False,-0.083333
1,CRM-1012,CAL-A13,2025-03-12T11:00:00Z,2025-03-11T16:30:00Z,True,18.500000
3,CRM-1018,CAL-A21,2025-03-17T10:20:00Z,2025-03-16T14:45:00Z,True,19.583333
2,CRM-1002,CAL-A2,2025-03-01T11:30:00Z,2025-02-28T15:45:00Z,True,19.750000
4,CRM-1011,CAL-A12,2025-03-07T10:15:00Z,2025-03-06T14:20:00Z,True,19.916667
9,CRM-1004,CAL-A4,2025-03-05T08:45:00Z,2025-03-04T12:00:00Z,True,20.750000
8,CRM-1014,CAL-A15,2025-03-20T14:00:00Z,2025-03-19T17:00:00Z,True,21.000000
14,CRM-1009,CAL-A10,2025-03-01T08:00:00Z,2025-02-28T11:00:00Z,True,21.000000


## Within-source duplicates

In [52]:
# calendar: same organizer, same day, overlapping attendee sets, close start
c = cal[["event_id", "title", "organizer", "day", "start", "attendees", "is_recurring"]]
sd = c.merge(c, on=["organizer", "day"], suffixes=("_a", "_b"))
sd = sd[sd["event_id_a"] < sd["event_id_b"]]
sd["dt_min"] = (sd["start_b"] - sd["start_a"]).dt.total_seconds().abs() / 60
sd["att_jaccard"] = [
    len(set(a) & set(b)) / max(1, len(set(a) | set(b))) for a, b in zip(sd["attendees_a"], sd["attendees_b"])
]
sd[["event_id_a", "title_a", "event_id_b", "title_b", "dt_min", "att_jaccard"]].sort_values("att_jaccard", ascending=False)

,event_id_a,title_a,event_id_b,title_b,dt_min,att_jaccard
5,CAL-A5,Investor Update - Pinnacle,CAL-A6,Pinnacle Group - Q1 Update,30.0,0.666667
20,CAL-A18,Weekly Team Sync,CAL-A8,Meridian Capital - Follow-up,360.0,0.200000


In [53]:
sd[(sd["att_jaccard"] >= 0.5) & (sd["dt_min"] <= 60)][["event_id_a", "title_a", "start_a", "event_id_b", "title_b", "start_b"]]

,event_id_a,title_a,start_a,event_id_b,title_b,start_b
5,CAL-A5,Investor Update - Pinnacle,2025-03-17 11:00:00,CAL-A6,Pinnacle Group - Q1 Update,2025-03-17 11:30:00


In [54]:
# recurring same-title rows are different weeks, not duplicates
cal[cal.duplicated("title", keep=False)][["event_id", "title", "start", "is_recurring"]]

,event_id,title,start,is_recurring
2,CAL-A3,Weekly Team Sync,2025-03-11 09:00:00,True
17,CAL-A18,Weekly Team Sync,2025-03-18 09:00:00,True


## Summary of issues found

In [55]:
issues = pd.DataFrame([
    ("crm", "CRM-1008", "meeting_date in MM-DD/YYYY"),
    ("crm", "CRM-1007", "meeting_time null; date-only match"),
    ("crm", "CRM-1010", "notes empty string"),
    ("crm", "CRM-1006/1009/1013/1019", "internal: client fields null"),
    ("crm", "CRM-1017", "client_name 'Multiple'; cal side has 3 client domains"),
    ("crm", "CRM-1005", "client_name with apostrophe; first.last key needs non-letters stripped"),
    ("crm", "CRM-1009", "status Cancelled, note says rescheduled to 3/26; no 3/26 counterpart anywhere"),
    ("crm", "CRM-1008 / CAL-A9", "Saturday meeting"),
    ("cal", "CAL-A4", "start/end UTC (Z) while others naive local; 19:00Z = 15:00 EDT but CRM says 14:00 (fixed -5 offset upstream?)"),
    ("cal", "CAL-A11", "end_time truncated to HH:MM; attendees empty; location/description null"),
    ("cal", "CAL-A16", "attendee 'raj.patel[at]atlasvc.com'"),
    ("cal", "CAL-A20", "attendee 'external-guests'"),
    ("cal", "CAL-A5 / CAL-A6", "same-source duplicate (Pinnacle, 30 min apart, overlapping)"),
    ("cal", "CAL-A3 / CAL-A18", "recurring series instances, not duplicates"),
    ("cal", "CAL-A3/A18/A19", "internal meetings with no crm record"),
    ("both", "status", "vocab differs: Completed/Scheduled/Cancelled vs confirmed/tentative; casing differs"),
    ("both", "location", "same room spelled 3 ways (HQ - prefix, floor/room suffix, Zoom URL suffix)"),
    ("both", "CRM-1013 / CAL-A14", "owner != organizer; only found by day+time+title pass"),
    ("both", "CRM-1001 / CAL-A1", "status Completed vs confirmed"),
    ("both", "CRM-1002 / CAL-A2", "In-Person NYC vs Zoom"),
    ("both", "CRM-1009 / CAL-A10", "Cancelled vs confirmed"),
    ("both", "CRM-1016 / CAL-A17", "13:00 vs 15:00"),
    ("both", "matched pairs", "calendar record created ~1 day before crm record in 14/17 pairs; ESG reversed"),
    ("both", "CRM-1003/1010/1020", "crm-only; clients never appear in any calendar attendee list"),
    ("both", "CAL-A3/A6/A11/A18/A19", "cal-only (A6 is the duplicate)"),
], columns=["source", "ids", "issue"])
issues

,source,ids,issue
0,crm,CRM-1008,meeting_date in MM-DD/YYYY
1,crm,CRM-1007,meeting_time null; date-only match
2,crm,CRM-1010,notes empty string
3,crm,CRM-1006/1009/1013/1019,internal: client fields null
4,crm,CRM-1017,client_name 'Multiple'; cal side has 3 client domains
5,crm,CRM-1005,client_name with apostrophe; first.last key needs non-le...
6,crm,CRM-1009,"status Cancelled, note says rescheduled to 3/26; no 3/26..."
7,crm,CRM-1008 / CAL-A9,Saturday meeting
8,cal,CAL-A4,start/end UTC (Z) while others naive local; 19:00Z = 15:...
9,cal,CAL-A11,end_time truncated to HH:MM; attendees empty; location/d...


## Checked, nothing notable

In [56]:
# ids unique and well-formed in both sources
crm["crm_id"].duplicated().sum(), cal["event_id"].duplicated().sum(), crm["crm_id"].str.match(r"^CRM-\d{4}$").all(), cal["event_id"].str.match(r"^CAL-A\d+$").all()

(np.int64(0), np.int64(0), np.True_, np.True_)

In [57]:
# no leading/trailing whitespace anywhere
def ws(df, skip=("attendees",)):
    cols = [c for c in df.columns if c not in skip and df[c].dtype == object or str(df[c].dtype) == "str"]
    return sum((df[c].notna() & (df[c].astype(str).str.strip() != df[c].astype(str))).sum() for c in cols if c not in skip)
ws(crm), ws(cal)

(np.int64(0), np.int64(0))

In [58]:
# only one non-ascii char (em dash in a note)
crm[crm["notes"].str.contains(r"[^\x00-\x7f]", na=False)][["crm_id", "notes"]]

,crm_id,notes
8,CRM-1009,Prep Q1 board materials. CANCELLED — rescheduled to 3/26.


In [59]:
# all meeting_time values that exist are HH:MM
crm["meeting_time"].dropna().str.len().value_counts()

meeting_time
5    19
Name: count, dtype: int64

In [60]:
# times sit on :00 or :30 in both sources
crm["start"].dt.minute.value_counts(), cal["start"].dt.minute.value_counts()

(start
 0.0     17
 30.0     2
 Name: count, dtype: int64,
 start
 0     19
 30     3
 Name: count, dtype: int64)

In [61]:
# durations are sane; no zero/negative, nothing spans midnight
cal["duration_min"].describe(), cal[(cal["duration_min"] <= 0) | (cal["start"].dt.date != cal["end"].dt.date)][["event_id", "start", "end"]]

(count     22.000000
 mean      75.681818
 std       36.589598
 min       15.000000
 25%       60.000000
 50%       75.000000
 75%       90.000000
 max      150.000000
 Name: duration_min, dtype: float64,
 Empty DataFrame
 Columns: [event_id, start, end]
 Index: [])

In [62]:
# recurring meetings are just shorter
cal.groupby("is_recurring")["duration_min"].describe()

,count,mean,std,min,25%,50%,75%,max
is_recurring,,,,,,,,
False,19.0,81.315789,35.817520,15.0,60.0,90.0,105.0,150.0
True,3.0,40.000000,17.320508,30.0,30.0,30.0,45.0,60.0


In [63]:
# date ranges overlap almost exactly
crm["date"].min(), crm["date"].max(), cal["start"].min(), cal["start"].max()

(Timestamp('2025-03-10 00:00:00'),
 Timestamp('2025-04-02 00:00:00'),
 Timestamp('2025-03-10 14:00:00'),
 Timestamp('2025-04-01 14:00:00'))

In [64]:
# created_at all valid ISO UTC; nothing created after its own meeting
crm["created"] = pd.to_datetime(crm["created_at"], utc=True).dt.tz_convert("America/New_York").dt.tz_localize(None)
cal["created"] = pd.to_datetime(cal["created_at"], utc=True).dt.tz_convert("America/New_York").dt.tz_localize(None)
(crm["created"] > crm["start"]).sum(), (cal["created"] > cal["start"]).sum()

(np.int64(0), np.int64(0))

In [65]:
# lead times unremarkable
pd.concat({"crm": (crm["start"] - crm["created"]).dt.days.describe(), "cal": (cal["start"] - cal["created"]).dt.days.describe()}, axis=1)

,crm,cal
count,19.000000,22.000000
mean,13.368421,18.227273
std,12.139058,19.692715
min,4.000000,0.000000
25%,8.000000,8.000000
50%,10.000000,10.500000
75%,13.500000,18.000000
max,58.000000,71.000000


In [66]:
# working hours only
crm["start"].dt.hour.value_counts().sort_index(), cal["start"].dt.hour.value_counts().sort_index()

(start
 8.0     1
 9.0     2
 10.0    4
 11.0    2
 12.0    1
 13.0    1
 14.0    4
 15.0    1
 16.0    2
 19.0    1
 Name: count, dtype: int64,
 start
 8     1
 9     4
 10    3
 11    2
 12    1
 14    4
 15    3
 16    2
 18    1
 19    1
 Name: count, dtype: int64)

In [67]:
# status casing is internally consistent per source
crm["status"].str.istitle().all(), cal["status"].str.islower().all()

(np.True_, np.True_)

In [68]:
# crm status does not encode past/future consistently; not useful for matching
crm.assign(past=crm["start"] < crm["created"].max()).groupby(["status", "past"]).size()

status     past 
Cancelled  True     1
Completed  True     1
Confirmed  False    8
           True     6
Scheduled  False    1
           True     1
Tentative  False    1
           True     1
dtype: int64

In [69]:
# tentative only on the empty reception
cal.groupby(["status", "is_recurring"]).size()

status     is_recurring
confirmed  False           18
           True             3
tentative  False            1
dtype: int64

In [70]:
# meeting_type agrees with location text within crm
crm.assign(loc_null=crm["location"].isna()).groupby(["meeting_type", "loc_null"]).size(), crm[(crm["meeting_type"] == "In-Person") & (crm["loc_kind"] == "virtual")]

(meeting_type  loc_null
 In-Person     False       9
 Internal      False       4
 Virtual       False       3
               True        4
 dtype: int64,
 Empty DataFrame
 Columns: [crm_id, subject, client_name, client_company, relationship_owner, meeting_date, meeting_time, meeting_type, location, notes, status, created_at, date, start, dow, client_key, client_in_cal, loc_kind, day, owner_email, created]
 Index: [])

In [71]:
# organizer always in attendees except the empty list
cal[[o not in a for o, a in zip(cal["organizer"], cal["attendees"])]][["event_id", "organizer", "attendees"]]

,event_id,organizer,attendees
10,CAL-A11,priya.sharma@firma.com,[]


In [72]:
# owner names map cleanly to firma.com emails
crm["relationship_owner"].map(name_key).isin(local_parts).all()

np.True_

In [73]:
# attendee counts small and unremarkable
cal["attendees"].map(len).value_counts().sort_index()

attendees
0    1
2    9
3    6
4    4
5    1
6    1
Name: count, dtype: int64

In [74]:
# crm subjects are generic but unique
crm["subject"].value_counts().max(), crm["subject"].str.len().describe()

(np.int64(1),
 count    20.000000
 mean     19.900000
 std       4.897905
 min      13.000000
 25%      15.000000
 50%      19.500000
 75%      25.000000
 max      29.000000
 Name: subject, dtype: float64)

In [75]:
# no crm same-owner same-start double booking
crm[crm.duplicated(["relationship_owner", "start"], keep=False)][["crm_id", "subject", "relationship_owner", "start"]]

,crm_id,subject,relationship_owner,start


In [76]:
# no crm same-client same-day duplicates
crm[crm.duplicated(["client_company", "day"], keep=False) & crm["client_company"].notna()][["crm_id", "subject", "client_company", "start"]]

,crm_id,subject,client_company,start
